In [1]:
from ultralytics import YOLO

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [3]:
model = YOLO('yolo11m.pt')

### Hız hesaplama mantığı

Araçların hızını ölçüyoruz. Mantık şöyle: yolun her iki tarafında kesikli çizgiler var ve bu çizgilerin arası **9.144 metre**. Aracın bir bölgeden diğerine gidene kadar geçen süreyi bulup bu mesafeye oranlayarak ortalama hızı buluyoruz.

Çizgilere tam denk gelmek zor olduğu için (aracın orta noktası 4-5 piksel kayabiliyor) çizgi yerine **dikdörtgen bölgeler** tanımlıyoruz. Araç bölgeye girdiğinde giriş anı kaydediliyor.

Sağdaki kesikli çizgi arası 86 piksel, soldaki 74 piksel. Bölge kullandığımız için aracın iki bölgeye girdiği noktalar arası dikey piksel farkını ölçüp oran-orantı yapıyoruz: sağda 86 piksel = 9.144 m ise, ölçtüğümüz `x` piksel = `(x*9.144)/86` metre.

**Önemli fark (eski koda göre):** Eski kodda süre `time.time()` (duvar saati) ile ölçülüyordu ve işlem FPS'ine bağlı olduğu için `30/fps` çarpanıyla telafi ediliyordu. Burada bunun yerine **frame numarası farkını video FPS'ine bölerek** süreyi videonun kendi zaman ekseninden okuyoruz:

`geçen_süre = (çıkış_frame - giriş_frame) / video_fps`

Bu yöntem işlem hızından tamamen bağımsızdır; `time.time()` ve `30/fps` çarpanına artık gerek yoktur.

In [18]:
import cv2
import numpy as np

kamera = cv2.VideoCapture('trafik.mp4')

video_fps = kamera.get(cv2.CAP_PROP_FPS)
print("Video FPS:", video_fps)

font = cv2.FONT_HERSHEY_SIMPLEX

region1 = np.array([(700,575),(1100,575),(1100,560),(700,560)]).reshape((-1,1,2))
region2 = np.array([(700,489),(1000,489),(1000,400),(700,400)]).reshape((-1,1,2))

region3 = np.array([(320,466),(580,466),(580,480),(320,480)]).reshape((-1,1,2))
region4 = np.array([(200,540),(550,540),(520,660),(70,660)]).reshape((-1,1,2))

uzunluk = 86
uzunluk2 = 74
distance = 9.144

Video FPS: 29.982105542289595


In [19]:
vehicles_entering = {}
vehicles_konum = {}
vehicles_speed = {}
vehicles_elapsed_time = {}

vehicles_entering2 = {}
vehicles_konum2 = {}
vehicles_speed2 = {}
vehicles_elapsed_time2 = {}

In [17]:
frame_no = 0
araclar = [2, 5, 7]
while True:
    ret, kare = kamera.read()
    if not ret:
        break

    frame_no += 1

    cv2.polylines(kare, [region2], True, (0,0,255), 2)
    cv2.polylines(kare, [region4], True, (0,0,255), 2)

    results = model.track(
        kare,
        persist=True,
        classes=araclar,
        conf=0.5,
        tracker='botsort.yaml',
        verbose=False
    )

    if results[0].boxes.id is None:
        cv2.imshow("kamera", kare)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    boxes = results[0].boxes.xywh.cpu().numpy()
    ids = results[0].boxes.id.cpu().numpy().astype(int)
    clss = results[0].boxes.cls.cpu().numpy().astype(int)

    for (cx, cy, w, h), isim, sinif in zip(boxes, ids, clss):
        arac_tipi = model.names[sinif]
        cx, cy = int(cx), int(cy)
        x = int(cx - w/2)
        y = int(cy - h/2)

        inside_region1 = cv2.pointPolygonTest(region1, (cx, cy), False)
        if inside_region1 > 0:
            if isim not in vehicles_entering:
                vehicles_entering[isim] = frame_no
                vehicles_konum[isim] = cy

        if isim in vehicles_entering:
            inside_region2 = cv2.pointPolygonTest(region2, (cx, cy), False)
            if inside_region2 > 0:
                if isim not in vehicles_elapsed_time:
                    frame_farki = frame_no - vehicles_entering[isim]
                    vehicles_elapsed_time[isim] = frame_farki / video_fps
                elapsed_time = vehicles_elapsed_time[isim]

                mesafe = vehicles_konum[isim] - cy

                if isim not in vehicles_speed and elapsed_time > 0:
                    a_speed_kh = (distance / elapsed_time) * 3.6
                    real_kh = a_speed_kh * mesafe / uzunluk
                    vehicles_speed[isim] = int(real_kh)

                if isim in vehicles_speed:
                    sag_hiz = str(vehicles_speed[isim]) + ' km/h'
                    cv2.putText(kare, sag_hiz, (x, y), font, 1.7, (0,150,255), 3)

        inside_region3 = cv2.pointPolygonTest(region3, (cx, cy), False)
        if inside_region3 > 0:
            if isim not in vehicles_entering2:
                vehicles_entering2[isim] = frame_no
                vehicles_konum2[isim] = cy

        if isim in vehicles_entering2:
            inside_region4 = cv2.pointPolygonTest(region4, (cx, cy), False)
            if inside_region4 > 0:
                if isim not in vehicles_elapsed_time2:
                    frame_farki2 = frame_no - vehicles_entering2[isim]
                    vehicles_elapsed_time2[isim] = frame_farki2 / video_fps
                elapsed_time2 = vehicles_elapsed_time2[isim]

                mesafe2 = -(vehicles_konum2[isim] - cy)

                if isim not in vehicles_speed2 and elapsed_time2 > 0:
                    a_speed_kh2 = (distance / elapsed_time2) * 3.6
                    real_kh2 = a_speed_kh2 * mesafe2 / uzunluk2
                    vehicles_speed2[isim] = int(real_kh2)

                if isim in vehicles_speed2:
                    sol_hiz = str(vehicles_speed2[isim]) + ' km/h'
                    cv2.putText(kare, sol_hiz, (x, y), font, 1.7, (0,255,255), 3)

    cv2.imshow("kamera", kare)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

kamera.release()
cv2.destroyAllWindows()